# LangSmith Sandboxes: the basics

A sandbox is an isolated machine an agent can use to run code, install packages and work with files, without touching your laptop, servers or data.

Each sandbox is a **Firecracker microVM** with its own kernel (stronger isolation than a container). It starts in under a second.

This notebook walks through:

1. Create a sandbox and look inside
2. Run code
3. Move files in and out
4. Preview an app running inside the sandbox
5. Keep secrets out of the sandbox
6. Control where it can connect
7. Lifecycle: stop, wake, and what survives
8. Snapshots: start from a ready-made environment

**Setup:** `LANGSMITH_API_KEY` (and `LANGSMITH_WORKSPACE_ID` if your key is org-scoped) in `../.env`.

In [ ]:
import time

from dotenv import load_dotenv
from langsmith.sandbox import SandboxClient

load_dotenv("../.env")
client = SandboxClient()

## 1. Create a sandbox and look inside

`create_sandbox()` boots one from the default snapshot (Ubuntu with common dev tools). `idle_ttl_seconds` means it cleans itself up if we forget.

In [ ]:
start = time.time()
sb = client.create_sandbox(idle_ttl_seconds=600)
print(f"Sandbox {sb.name} ready in {time.time() - start:.2f}s")

In [ ]:
for cmd in ["uname -sr", "grep PRETTY_NAME /etc/os-release", "whoami", "nproc", "free -h | head -2", "python3 --version"]:
    print(f"$ {cmd}\n{sb.run(cmd).stdout.strip()}\n")

The agent gets **root** inside its own VM. That's the point: it can install packages and run anything, and the blast radius is the sandbox.

## 2. Run code

This is what an agent's `execute` tool does under the hood: send a command, get stdout, stderr and the exit code back.

In [ ]:
agent_written_code = '''
import statistics
revenue = [6.61, 6.60, 6.57, 6.64, 6.61, 6.58]
print(f"mean={statistics.mean(revenue):.2f}B  stdev={statistics.stdev(revenue):.3f}B")
'''
result = sb.run(f"python3 - <<'EOF'\n{agent_written_code}\nEOF")
print("exit code:", result.exit_code)
print(result.stdout)

In [ ]:
# Errors come back as data, so the agent can read them and fix its own code.
result = sb.run("python3 -c 'import pandas'")
print("exit code:", result.exit_code)
print(result.stderr.strip().splitlines()[-1])

In [ ]:
# The default image's Python is OS-managed (PEP 668), hence --break-system-packages.
# In production, bake packages into a snapshot instead (section 7).
start = time.time()
sb.run("pip install --break-system-packages --quiet pandas matplotlib", timeout=300)
print(f"installed in {time.time() - start:.0f}s:", sb.run("python3 -c 'import pandas; print(pandas.__version__)'").stdout.strip())

## 3. Move files in and out

Two ways files move:

- **Agent tools** (`read_file`, `write_file`, `execute`) run *inside* the sandbox during the agent loop.
- **File transfer APIs** (`write`, `read`, download URLs) are for *your code*: seed data before the agent runs, collect results after.

Here our code seeds a CSV, "the agent" charts it, and we take the chart back.

In [ ]:
csv = b'''region,year,revenue_b
AFRICA,1996,6.61
AMERICA,1996,6.62
ASIA,1996,6.69
EUROPE,1996,6.75
MIDDLE EAST,1996,6.61
'''
sb.write("/workspace/data/revenue.csv", csv)

chart_script = '''
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
df = pd.read_csv("/workspace/data/revenue.csv")
ax = df.plot.bar(x="region", y="revenue_b", legend=False, title="Revenue by region, 1996 ($B)")
ax.set_ylim(6.4, 6.8)
plt.tight_layout()
plt.savefig("/workspace/out/chart.png", dpi=120)
'''
sb.run("mkdir -p /workspace/out")
sb.write("/workspace/chart.py", chart_script.encode())
print(sb.run("python3 /workspace/chart.py && ls -la /workspace/out").stdout)

In [ ]:
from IPython.display import Image, display

png = sb.read("/workspace/out/chart.png")
display(Image(png))

In [ ]:
# Or hand someone a signed, expiring download link (no LangSmith login needed).
link = sb.generate_download_url("/workspace/out/chart.png", expires_in_seconds=900)
print(link.download_url)

## 4. Preview an app running inside the sandbox

When an agent builds something that *runs* (a web app, an API, a notebook server), a **service URL** lets a person or a test reach it, without running AI-written code on your own machine.

The link carries a signed, expiring token (10 minutes by default, 24 hours max). Anyone holding an unexpired link can open it, no LangSmith login needed, so treat it like an "anyone with the link" share. For a finished *file*, use a download link instead (section 3). Browsers open download links with scripts disabled, so anything interactive (charts, apps) needs a service URL.

In [ ]:
html = b'''<html><body style="font-family:sans-serif;margin:40px">
<h1>Revenue report</h1><p>Generated inside a LangSmith sandbox.</p>
<img src="chart.png" width="600"></body></html>'''
sb.write("/workspace/out/index.html", html)

server = sb.run("python3 -m http.server 8000 --directory /workspace/out", timeout=0, wait=False)
time.sleep(1)
service = sb.service(port=8000, expires_in_seconds=900)
print("Open in a browser (expires in 15 min):")
print(service.browser_url)
print("\nProgrammatic check:", service.get("/").status_code)

## 5. Keep secrets out of the sandbox

Anything inside a sandbox can be read by the code running there, including an agent that's been prompt-injected. So **credentials should never live inside it**.

Instead, an **auth proxy** sits *outside* the sandbox and adds credentials to outbound requests on the way out. Here, the code inside calls an authenticated API (LangSmith's), and there is no key anywhere in the sandbox.

In [ ]:
secure = client.create_sandbox(access_delegation={"mode": "INHERIT"}, idle_ttl_seconds=300)

print("What the code inside can see:")
print(secure.run("env | grep -i -E 'key|token|secret'").stdout)

status = secure.run(
    "curl -s -o /dev/null -w '%{http_code}' https://api.smith.langchain.com/api/v1/sessions?limit=1"
).stdout
print("Call to the LangSmith API from inside the sandbox ->", status)

The only "key" in the sandbox is a **placeholder**. The request succeeds because the proxy attached the real credential after it left the sandbox.

The same proxy works for any API. Store the secret as a LangSmith workspace secret and add a rule, for example:

```python
proxy_config={"rules": [{
    "name": "openai",
    "match_hosts": ["api.openai.com"],
    "headers": [{"name": "Authorization", "type": "workspace_secret", "value": "Bearer {OPENAI_API_KEY}"}],
}]}
```

S3, GCS and git **mounts** use it too, so bucket credentials never enter the sandbox.

## 6. Control where it can connect

By default this organization's sandboxes can reach the whole internet. An allow list turns that into default-deny, which closes off data exfiltration paths.

In [ ]:
locked = client.create_sandbox(
    proxy_config={"access_control": {"allow_list": ["pypi.org:443", "files.pythonhosted.org:443"]}},
    idle_ttl_seconds=300,
)
for url in ["https://pypi.org/simple/", "https://example.com"]:
    r = locked.run(f"curl -s -o /dev/null -w '%{{http_code}}' --max-time 8 {url}")
    print(f"{url:30} -> {'allowed (' + r.stdout + ')' if r.exit_code == 0 else 'blocked'}")

## 7. Lifecycle: stop, wake, and what survives

```text
running ──(idle_ttl_seconds, default 10 min)──▶ stopped ──(delete_after_stop_seconds, default 14 days)──▶ deleted
```

A stopped sandbox costs nothing to keep and wakes on the next command, file operation or URL request. The **disk survives** a stop; **running processes don't** (the VM boots fresh).

In [ ]:
sb.write("/workspace/notes.txt", b"the agent's working notes")
print("web server from section 4 running:", "http.server" in sb.run("ps aux").stdout)

sb.stop()
print("status:", client.get_sandbox(sb.name).status)

start = time.time()
notes = sb.run("cat /workspace/notes.txt").stdout.strip()
print(f"notes.txt: {notes!r} (woke in {time.time() - start:.1f}s)")
print("web server still running:", "http.server" in sb.run("ps aux").stdout)
print("status:", client.get_sandbox(sb.name).status)

In [ ]:
vars(sb)

So an agent should treat its sandbox as **scratch space**, and put anything that must outlive the conversation somewhere durable: your storage, a repo, or a download link to a finished file.

## 8. Snapshots: start from a ready-made environment

Installing packages on every run is slow. Capture a configured sandbox as a **snapshot** (or build one from a Docker image) and boot new sandboxes from it. Managed Deep Agents does this automatically from `sandbox/setup.sh`.

In [ ]:
snapshot_name = f"analytics-demo-{int(time.time())}"
snap = sb.capture_snapshot(snapshot_name, timeout=300) # sb still has the pandas we pip-installed in section 2
print("captured:", snap.name)

start = time.time()
fresh = client.create_sandbox(snapshot_name=snapshot_name, idle_ttl_seconds=300)
print(f"booted from snapshot in {time.time() - start:.2f}s")
print("pandas already installed:", fresh.run("python3 -c 'import pandas; print(pandas.__version__)'").stdout.strip())

## Clean up

In [ ]:
for box in (sb, secure, locked, fresh):
    box.delete()
# Sandbox deletion finishes in the background; the snapshot frees up a few seconds later.
for _ in range(30):
    try:
        client.delete_snapshot(snap.id)
        break
    except Exception:
        time.sleep(2)
print("deleted sandboxes and snapshot")

## Where this goes next

The same sandbox becomes an agent's workspace. In `../mda-snowflake-analyst/`, a Managed Deep Agent queries Snowflake through LangSmith Tools (as the signed-in user), writes and runs analysis code in its own sandbox, and returns a chart and a shareable report.